# DulceBot con Llama: Fine-tuning LoRA + RAG + Evaluación

Este notebook es la pieza de **fine-tuning y evaluación** del proyecto [Antojo de Yanet](https://antojo-yanet.pages.dev/). Ajusta un modelo de la familia **Llama** (TinyLlama-1.1B-Chat) con **LoRA** para que responda con la voz y las reglas de DulceBot, usando como contexto la **misma base de conocimiento RAG** que usa la app en producción.

| Pieza del Módulo 1 | Dónde aparece aquí |
|---|---|
| Masterclass 1: Llama | TinyLlama-1.1B-Chat, arquitectura Llama 2 |
| Masterclass 2: RAG | Documentos reales de `api/docs/`, embeddings `paraphrase-multilingual-MiniLM-L12-v2` y umbral 0.25, igual que `api/src/lib/rag.ts` |
| Masterclass 3: LoRA y evaluación | LoRA sobre `q/k/v/o_proj`, pérdida solo sobre la respuesta y comparación base vs. ajustado con LLM-como-juez |
| E-learning: pipeline | pregunta → RAG → prompt → modelo ajustado → evaluación → adaptador guardado |

**Antes de empezar:**
1. *Entorno de ejecución → Cambiar tipo de entorno → GPU T4.*
2. En *Secrets* (ícono de llave) agrega `GROQ_API_KEY` (para el juez) y, opcionalmente, `HF_TOKEN`.

**Por qué no se ajusta con el menú:** el catálogo (`menu.md`) se regenera desde la base de datos cada vez que cambia un producto o un precio. Un modelo ajustado con esos datos quedaría desactualizado al día siguiente. Por eso el menú se consulta con RAG y herramientas, y el fine-tuning se usa para lo que no cambia: **tono, formato y políticas del negocio**.

## 1. Configuración del entorno

In [ ]:
# Install dependencies (torchao from the Colab image conflicts with peft, as seen in Clase 3)
!pip uninstall -y torchao --quiet
!pip install -q transformers peft accelerate datasets sentence-transformers groq

In [ ]:
# Read API keys from Colab Secrets and check the GPU
import os, re, json, time, unicodedata
import requests
import numpy as np
import torch
from google.colab import userdata
from groq import Groq

try:
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    print("Sesión de Hugging Face iniciada.")
except Exception:
    print("Sin HF_TOKEN: no hace falta para TinyLlama (modelo abierto).")

groq_client = Groq(api_key=userdata.get("GROQ_API_KEY"))
MODELO_JUEZ = "openai/gpt-oss-20b"  # same model DulceBot uses in production

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO DISPONIBLE: activa la GPU T4")

## 2. Base de conocimiento (RAG)

Descargamos los documentos reales del repositorio y los partimos en secciones `#`/`##`, igual que `cargarDocumentos()` en la API. La recuperación usa similitud coseno con el mismo modelo de embeddings y el mismo umbral que producción, y toma los 3 fragmentos más relevantes (producción usa hasta 5).

In [ ]:
# Download the knowledge base from the public repository and split it into chunks
URL_DOCS = "https://raw.githubusercontent.com/MarioSimbron/antojo-yanet/main/api/docs/"
DOCUMENTOS = ["faq.md", "horarios.md", "politicas-encargos.md"]


def cargar_chunks(nombres):
    """Downloads each markdown document and splits it at h1/h2 boundaries, keeping
    only sections longer than 80 characters (same rule as rag.ts).

    @author Mario Simbron Gonzalez <simbron420@gmail.com>
    @param nombres: list of file names inside api/docs/.
    @returns: list of dicts with "fuente" (file name) and "texto" (section text).
    """
    chunks = []
    for nombre in nombres:
        texto = requests.get(URL_DOCS + nombre, timeout=30).text
        for seccion in re.split(r"\n(?=#{1,2} )", texto):
            if len(seccion.strip()) > 80:
                chunks.append({"fuente": nombre, "texto": seccion.strip()})
    return chunks


chunks = cargar_chunks(DOCUMENTOS)
print(f"{len(chunks)} fragmentos cargados de {len(DOCUMENTOS)} documentos")

In [ ]:
# Embed the chunks and define the retrieval function
from sentence_transformers import SentenceTransformer

modelo_embeddings = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings_chunks = modelo_embeddings.encode([c["texto"] for c in chunks], normalize_embeddings=True)

UMBRAL_SEMANTICO = 0.25        # same threshold as UMBRAL_SEMANTICO in rag.ts
TOP_K = 3                      # production retrieves up to 5; 3 keeps TinyLlama prompts short
MAX_CARACTERES_CONTEXTO = 700  # per chunk; keeps prompt + answer under the training max length


def buscar_contexto(pregunta):
    """Returns the TOP_K most similar knowledge-base chunks for a question, joined,
    or an empty string when no chunk clears the similarity threshold (greetings,
    off-topic). Run 1 used only the single best chunk and "¿A qué hora abren el
    domingo?" retrieved "Días festivos" instead of "Horario de atención".

    @author Mario Simbron Gonzalez <simbron420@gmail.com>
    @param pregunta: the customer's question.
    @returns: the chunk texts (each truncated to MAX_CARACTERES_CONTEXTO) or "".
    """
    q = modelo_embeddings.encode([pregunta], normalize_embeddings=True)[0]
    similitudes = embeddings_chunks @ q
    mejores = [i for i in np.argsort(-similitudes)[:TOP_K] if similitudes[i] >= UMBRAL_SEMANTICO]
    return "\n\n".join(chunks[i]["texto"][:MAX_CARACTERES_CONTEXTO] for i in mejores)


for p in ["¿A qué hora abren el domingo?", "¿Me regresan el depósito si cancelo?", "¿Quién ganó el partido?"]:
    print(f"\n{p}\n→ {buscar_contexto(p)[:120] or '(sin contexto: debajo del umbral)'}")

## 3. Dataset de DulceBot

Cada tema tiene:
- **4 preguntas de entrenamiento** (paráfrasis de cómo escriben los clientes).
- **1 pregunta de prueba que el modelo nunca ve al entrenar**, para medir si generaliza.
- La **respuesta de referencia** con la voz de DulceBot (amigable, cálida, directa, texto plano y sin emojis), basada en los documentos reales.
- Los **datos clave** que una respuesta correcta debe mencionar (métrica objetiva).

In [ ]:
# DulceBot training/evaluation dataset grounded in faq.md, horarios.md and politicas-encargos.md
TEMAS = [
    {
        "tema": "Horario domingo",
        "entrenamiento": ["¿A qué hora abren el domingo?", "¿Abren los domingos?",
                          "¿Cuál es su horario del domingo?", "¿Hasta qué hora están abiertos el domingo?"],
        "prueba": "Este domingo quiero pasar temprano, ¿a qué hora abren y hasta qué hora están?",
        "respuesta": "Los domingos abrimos de 8:00 a.m. a 3:00 p.m. ¿Te ayudo a armar tu pedido?",
        "datos_clave": ["8:00", "3:00"],
    },
    {
        "tema": "Horario entre semana",
        "entrenamiento": ["¿Cuál es su horario?", "¿A qué hora abren entre semana?",
                          "¿A qué hora cierran los sábados?", "¿Qué horario tienen de lunes a viernes?"],
        "prueba": "Salgo del trabajo a las 8 de la noche un martes, ¿todavía alcanzo a pasar?",
        "respuesta": "De lunes a viernes abrimos de 7:00 a.m. a 9:00 p.m. y los sábados de 7:00 a.m. a 10:00 p.m. ¿Te puedo ayudar con algo más?",
        "datos_clave": ["9:00"],
    },
    {
        "tema": "Costo de envío",
        "entrenamiento": ["¿Cuánto cuesta el envío?", "¿Cobran envío a domicilio?",
                          "¿Hay monto mínimo para pedir a domicilio?", "¿Cuánto me cobran por llevarme el pedido a mi casa?"],
        "prueba": "Si solo pido dos conchas, ¿me las llevan y cuánto sale el envío?",
        "respuesta": "El envío a domicilio cuesta $30 MXN fijo, sin importar el tamaño del pedido, y no hay monto mínimo. ¿Te ayudo a hacer tu pedido?",
        "datos_clave": ["30"],
    },
    {
        "tema": "Tiempo de entrega",
        "entrenamiento": ["¿Cuánto tarda la entrega?", "¿En cuánto tiempo llega mi pedido?",
                          "¿Qué tan rápido entregan a domicilio?", "¿Cuánto se tardan en llevarlo?"],
        "prueba": "Si pido ahorita, ¿como en cuánto me llega a la casa?",
        "respuesta": "La entrega a domicilio tarda de 30 a 60 minutos, según la zona y la demanda del día. ¿Quieres hacer un pedido?",
        "datos_clave": ["30", "60"],
    },
    {
        "tema": "Anticipación de encargos",
        "entrenamiento": ["¿Con cuánto tiempo debo hacer un encargo?", "¿Cuántos días antes tengo que pedir un pastel?",
                          "¿Puedo encargar un pastel para mañana?", "¿Cuál es la anticipación mínima para un encargo?"],
        "prueba": "Quiero un pastel para la fiesta de mi hija en dos meses, ¿ya lo puedo encargar?",
        "respuesta": "Los encargos se hacen con al menos 48 horas y máximo 30 días de anticipación. ¿Qué te gustaría encargar?",
        "datos_clave": ["48", "30 dias"],
    },
    {
        "tema": "Depósito",
        "entrenamiento": ["¿Cuánto tengo que dejar de anticipo?", "¿Cómo se paga un encargo?",
                          "¿Piden depósito para los pasteles?", "¿Tengo que pagar todo el pastel por adelantado?"],
        "prueba": "Para apartar mi rosca, ¿cuánto dinero tengo que dar primero?",
        "respuesta": "Para confirmar un encargo pagas un depósito del 50% y el resto lo liquidas al recibirlo o recogerlo. ¿Te ayudo a iniciar tu encargo?",
        "datos_clave": ["50%"],
    },
    {
        "tema": "Cancelación",
        "entrenamiento": ["¿Qué pasa si cancelo mi encargo?", "¿Me regresan el depósito si cancelo?",
                          "¿Puedo cancelar un pastel que ya aparté?", "¿Cuál es la política de cancelación?"],
        "prueba": "Ya no voy a necesitar el pastel que encargué para dentro de una semana, ¿pierdo mi dinero?",
        "respuesta": "Si cancelas con más de 72 horas de anticipación recibes un cupón por el valor de tu depósito, válido por 90 días. Con menos de 72 horas el depósito no es reembolsable.",
        "datos_clave": ["72 horas", "cupon"],
    },
    {
        "tema": "Formas de pago",
        "entrenamiento": ["¿Qué formas de pago aceptan?", "¿Aceptan tarjeta?",
                          "¿Puedo pagar en efectivo?", "¿Cómo puedo pagar mi pedido?"],
        "prueba": "No traigo efectivo, ¿puedo pagar con mi tarjeta de crédito?",
        "respuesta": "Aceptamos efectivo y tarjeta de débito o crédito; lo eliges al hacer tu pedido. ¿Algo más en lo que te ayude?",
        "datos_clave": ["tarjeta"],
    },
    {
        "tema": "Pedido sin cuenta",
        "entrenamiento": ["¿Puedo pedir sin cuenta?", "¿Tengo que registrarme para comprar?",
                          "¿Es obligatorio crear una cuenta?", "¿Puedo comprar como invitado?"],
        "prueba": "No quiero darme de alta en la página, ¿así puedo hacer mi pedido?",
        "respuesta": "Sí, puedes pedir como invitado con tu nombre, correo y teléfono. Si después creas una cuenta con ese correo, tus pedidos se asocian solos.",
        "datos_clave": ["invitado"],
    },
    {
        "tema": "Puntos de lealtad",
        "entrenamiento": ["¿Cómo gano puntos?", "¿Cómo funciona el programa de puntos?",
                          "¿Cuánto valen los puntos?", "¿Los pedidos de invitado acumulan puntos?"],
        "prueba": "Compro seguido con ustedes, ¿me dan algún beneficio por ser cliente frecuente?",
        "respuesta": "Ganas 1 punto por cada $10 MXN de compra entregada, solo con cuenta registrada. Con 100 puntos obtienes $10 MXN de descuento.",
        "datos_clave": ["punto", "100"],
    },
    {
        "tema": "Factura",
        "entrenamiento": ["¿Dan factura?", "¿Puedo facturar mi compra?",
                          "¿Cómo solicito mi factura?", "¿Qué datos necesitan para facturar?"],
        "prueba": "Necesito deducir la compra de mi empresa, ¿me pueden facturar?",
        "respuesta": "Sí, puedes pedir factura al hacer tu pedido o después desde el seguimiento, con tu RFC, razón social y uso de CFDI.",
        "datos_clave": ["rfc"],
    },
    {
        "tema": "Encargo y stock juntos",
        "entrenamiento": ["¿Puedo pedir un pastel de encargo y pan en el mismo pedido?", "¿Puedo juntar un encargo con productos normales?",
                          "¿Por qué no puedo pagar mi rosca junto con las conchas?", "¿Los encargos van separados?"],
        "prueba": "Quiero mi pastel encargado y de una vez unas donas, ¿lo meto todo en un solo pedido?",
        "respuesta": "No, los encargos van en un pedido separado de los productos de inventario. El sistema te ayuda a dividirlos si agregas ambos al carrito.",
        "datos_clave": ["separado"],
    },
    {
        "tema": "Personalizar pastel",
        "entrenamiento": ["¿Puedo personalizar mi pastel?", "¿Le pueden poner un mensaje al pastel?",
                          "¿Puedo elegir el sabor del pastel?", "¿Hacen pasteles con diseño especial?"],
        "prueba": "Quiero un pastel azul que diga 'Feliz cumple Ana', ¿se puede?",
        "respuesta": "Sí, en el mensaje personalizado del encargo puedes indicar sabor, color, texto y cualquier detalle. El equipo te contacta si hace falta aclarar algo.",
        "datos_clave": ["mensaje"],
    },
    {
        "tema": "Fuera de área",
        "entrenamiento": ["¿Quién ganó el partido ayer?", "¿Me ayudas con mi tarea de matemáticas?",
                          "¿Qué opinas de la política?", "Cuéntame un chiste"],
        "prueba": "¿Qué clima va a hacer mañana?",
        "respuesta": "Eso está fuera de mi área. ¿En qué puedo ayudarte con la panadería?",
        "datos_clave": ["fuera de mi area"],
    },
]

print(f"{len(TEMAS)} temas · {sum(len(t['entrenamiento']) for t in TEMAS)} ejemplos de entrenamiento · {len(TEMAS)} preguntas de prueba no vistas")

## 4. Modelo base (Llama) y línea base

Cargamos TinyLlama en `float16`, igual que en la Clase 3. Antes de entrenar generamos las respuestas del modelo base para las 14 preguntas de prueba. **El prompt es idéntico para ambos modelos** (system prompt de DulceBot + contexto RAG + pregunta), así que la única diferencia que se mide es el ajuste LoRA.

In [ ]:
# Load the Llama-family base model and its tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from transformers.utils import logging
logging.set_verbosity_error()
set_seed(42)

MODELO_BASE = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # official Meta alternative: "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
modelo = AutoModelForCausalLM.from_pretrained(MODELO_BASE, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", MODELO_BASE)

In [ ]:
# Prompt builder and deterministic generation shared by the base and fine-tuned models
SYSTEM_DULCEBOT = (
    "Eres DulceBot, la asistente virtual de la panadería Antojo de Yanet. "
    "Respondes de forma amigable, cálida y directa, en texto plano, sin emojis ni markdown, en una o dos oraciones. "
    "Usa solo la información del contexto; si el tema no es de la panadería, dilo y redirige."
)


def construir_prompt(pregunta):
    """Builds the chat prompt (DulceBot system prompt + retrieved RAG context +
    customer question) with TinyLlama's chat template.

    @author Mario Simbron Gonzalez <simbron420@gmail.com>
    @param pregunta: the customer's question.
    @returns: the formatted prompt string, ending at the assistant turn.
    """
    contexto = buscar_contexto(pregunta) or "Sin contexto disponible."
    mensajes = [
        {"role": "system", "content": SYSTEM_DULCEBOT},
        {"role": "user", "content": f"Contexto:\n{contexto}\n\nCliente: {pregunta}"},
    ]
    return tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)


def generar_respuesta(modelo_a_usar, pregunta, max_new_tokens=90):
    """Generates a greedy (deterministic) answer so base and fine-tuned outputs are
    comparable run after run.

    @author Mario Simbron Gonzalez <simbron420@gmail.com>
    @param modelo_a_usar: the base or LoRA model.
    @param pregunta: the customer's question.
    @param max_new_tokens: generation budget.
    @returns: the decoded answer without the prompt echo.
    """
    modelo_a_usar.eval()
    entrada = tokenizer(construir_prompt(pregunta), return_tensors="pt").to(modelo_a_usar.device)
    with torch.no_grad():
        salida = modelo_a_usar.generate(
            **entrada,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    nuevos = salida[0][entrada["input_ids"].shape[1]:]
    return tokenizer.decode(nuevos, skip_special_tokens=True).strip()


respuestas_base = {}
for t in TEMAS:
    respuestas_base[t["tema"]] = generar_respuesta(modelo, t["prueba"])
    print(f"--- {t['tema']}\nCliente: {t['prueba']}\nBase: {respuestas_base[t['tema']]}\n")

## 5. Fine-tuning con LoRA

Usamos la configuración del notebook final de la Clase 3:
- LoRA de rango 16 sobre las cuatro proyecciones de atención.
- La pérdida se calcula **solo sobre la respuesta**. Las etiquetas del prompt se enmascaran con `-100`, así el modelo no gasta capacidad en memorizar el contexto, solo aprende a responder.

In [ ]:
# Build the tokenized training set with the loss masked on the prompt tokens
from datasets import Dataset

MAX_LONGITUD = 1280

ejemplos = [
    {"prompt": construir_prompt(pregunta), "respuesta": t["respuesta"]}
    for t in TEMAS
    for pregunta in t["entrenamiento"]
]


def tokenizar_ejemplo(ejemplo):
    """Tokenizes prompt + answer and masks the prompt labels with -100 so the loss
    only measures how well the model reproduces DulceBot's answer.

    @author Mario Simbron Gonzalez <simbron420@gmail.com>
    @param ejemplo: dict with "prompt" and "respuesta".
    @returns: dict with input_ids, attention_mask and labels.
    """
    prompt_ids = tokenizer(ejemplo["prompt"], add_special_tokens=True)["input_ids"]
    respuesta_ids = tokenizer(ejemplo["respuesta"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
    input_ids = (prompt_ids + respuesta_ids)[:MAX_LONGITUD]
    labels = ([-100] * len(prompt_ids) + respuesta_ids)[:MAX_LONGITUD]
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}


dataset = Dataset.from_list(ejemplos).shuffle(seed=42)
dataset_tokenizado = dataset.map(tokenizar_ejemplo, remove_columns=dataset.column_names)
longitudes = [len(x) for x in dataset_tokenizado["input_ids"]]
print(f"{len(dataset_tokenizado)} ejemplos · longitud máxima {max(longitudes)} tokens (límite {MAX_LONGITUD})")

In [ ]:
# Configure LoRA and attach it to the base model
from peft import LoraConfig, get_peft_model

config_lora = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()

In [ ]:
# Train
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

args_entrenamiento = TrainingArguments(
    output_dir="/content/resultados_dulcebot",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,  # effective batch of 4 with the longer top-3 prompts
    learning_rate=1e-4,
    logging_steps=2,
    save_strategy="no",
    report_to="none",
    fp16=True,
)
trainer = Trainer(
    model=modelo_lora,
    args=args_entrenamiento,
    train_dataset=dataset_tokenizado,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=modelo_lora, padding=True, label_pad_token_id=-100),
)
resultado_entrenamiento = trainer.train()

In [ ]:
# Plot the training loss curve
import matplotlib.pyplot as plt

historial = [(x["step"], x["loss"]) for x in trainer.state.log_history if "loss" in x]
pasos, perdidas = zip(*historial)
print(f"Loss inicial: {perdidas[0]:.3f} · loss final: {perdidas[-1]:.3f} · reducción: {(1 - perdidas[-1] / perdidas[0]) * 100:.0f}%")
print("Recuerda: la loss no es % de aciertos. Solo indica que el modelo se acercó a las respuestas del dataset.")

plt.figure(figsize=(6, 3.2))
plt.plot(pasos, perdidas, marker="o", markersize=3)
plt.title("Pérdida durante el entrenamiento LoRA")
plt.xlabel("Paso")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Evaluación: misma prueba, dos modelos

Como en la Clase 3, comparamos el modelo base y el ajustado con **las mismas 14 preguntas que ninguno vio al entrenar** y dos métricas:
1. **Datos clave (objetiva):** porcentaje de datos obligatorios que menciona la respuesta (horario, costo, porcentaje, etc.).
2. **LLM-como-juez:** `gpt-oss-20b` en Groq califica del 1 al 5 la exactitud, el tono y el formato, con una rúbrica basada en las reglas de DulceBot en producción.

In [ ]:
# Generate the fine-tuned answers for the held-out questions
respuestas_lora = {}
for t in TEMAS:
    respuestas_lora[t["tema"]] = generar_respuesta(modelo_lora, t["prueba"])
    print(f"--- {t['tema']}\nCliente: {t['prueba']}\nBase: {respuestas_base[t['tema']]}\nLoRA: {respuestas_lora[t['tema']]}\n")

In [ ]:
# Objective metric: share of mandatory facts mentioned in each answer


def normalizar(texto):
    """Lowercases and strips accents so fact matching ignores spelling variants.

    @author Mario Simbron Gonzalez <simbron420@gmail.com>
    @param texto: any string.
    @returns: the normalized string.
    """
    sin_acentos = unicodedata.normalize("NFD", texto).encode("ascii", "ignore").decode()
    return sin_acentos.lower()


def cobertura_datos(respuesta, datos_clave):
    """Fraction of the topic's key facts that appear in the answer.

    @author Mario Simbron Gonzalez <simbron420@gmail.com>
    @param respuesta: model answer.
    @param datos_clave: list of mandatory facts.
    @returns: float between 0 and 1.
    """
    r = normalizar(respuesta)
    return sum(normalizar(d) in r for d in datos_clave) / len(datos_clave)


for nombre, respuestas in [("Base", respuestas_base), ("LoRA", respuestas_lora)]:
    cobertura = np.mean([cobertura_datos(respuestas[t["tema"]], t["datos_clave"]) for t in TEMAS])
    print(f"{nombre}: {cobertura * 100:.0f}% de datos clave mencionados")

In [ ]:
# LLM-as-judge with a rubric derived from DulceBot's production rules
PROMPT_JUEZ = """Eres un juez de calidad para DulceBot, la asistente virtual de la panadería Antojo de Yanet.

Recibes:
- PREGUNTA DEL CLIENTE: {pregunta}
- CONTEXTO (la única fuente válida): {contexto}
- RESPUESTA DE REFERENCIA (escrita por el equipo): {referencia}
- RESPUESTA A CALIFICAR: {respuesta}

Califica cada criterio del 1 al 5:
1) exactitud: ¿usa los datos correctos del contexto sin inventar nada? 5 = correcta y completa; 3 = parcial; 1 = inventa o contradice el contexto. Si la pregunta es ajena a la panadería y la respuesta lo reconoce y redirige, vale 5.
2) tono: ¿suena a DulceBot (amigable, cálida, directa, de tú)? 5 = cumple; 1 = fría, robótica o extraña.
3) formato: texto plano sin emojis ni markdown, de 1 a 3 oraciones, sin texto basura ni repeticiones. 5 = cumple; 1 = larga, repetida o con basura.

Devuelve ÚNICAMENTE un JSON válido, sin texto antes ni después:
{{"exactitud": <1-5>, "tono": <1-5>, "formato": <1-5>, "comentario": "<una frase>"}}"""


def calificar(pregunta, respuesta, referencia):
    """Asks the Groq judge to score an answer; retries with back-off because the free
    tier limits tokens per minute.

    @author Mario Simbron Gonzalez <simbron420@gmail.com>
    @param pregunta: held-out customer question.
    @param respuesta: answer to score.
    @param referencia: reference answer written by the team.
    @returns: dict with exactitud, tono, formato (1-5) and comentario; scores are None if every attempt failed.
    """
    prompt = PROMPT_JUEZ.format(
        pregunta=pregunta,
        contexto=buscar_contexto(pregunta) or "Sin contexto disponible.",
        referencia=referencia,
        respuesta=respuesta or "(respuesta vacía)",
    )
    for intento in range(1, 5):
        try:
            r = groq_client.chat.completions.create(
                model=MODELO_JUEZ,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )
            return json.loads(re.search(r"\{.*\}", r.choices[0].message.content, re.S).group(0))
        except Exception as e:
            print(f"   reintento {intento} ({type(e).__name__}), esperando {20 * intento}s…")
            time.sleep(20 * intento)
    return {"exactitud": None, "tono": None, "formato": None, "comentario": "sin calificación"}


filas = []
for t in TEMAS:
    for nombre, respuestas in [("Base", respuestas_base), ("LoRA", respuestas_lora)]:
        nota = calificar(t["prueba"], respuestas[t["tema"]], t["respuesta"])
        filas.append({
            "tema": t["tema"],
            "modelo": nombre,
            "datos_clave": cobertura_datos(respuestas[t["tema"]], t["datos_clave"]),
            **nota,
        })
        print(f"{t['tema']:<26} {nombre:<5} exactitud={nota['exactitud']} tono={nota['tono']} formato={nota['formato']}")
        time.sleep(3)

In [ ]:
# Results table and comparison chart
import pandas as pd

resultados = pd.DataFrame(filas)
criterios = ["exactitud", "tono", "formato"]
resultados[criterios] = resultados[criterios].apply(pd.to_numeric, errors="coerce")  # failed judge calls → NaN
resumen = resultados.groupby("modelo")[criterios].mean().round(2)
resumen["datos_clave_%"] = (resultados.groupby("modelo")["datos_clave"].mean() * 100).round(0)
display(resumen)

ax = resumen[criterios].T.plot(kind="bar", figsize=(6, 3.5), rot=0, ylim=(0, 5), color=["#b0a89f", "#c0562f"])
ax.set_title("DulceBot: modelo base vs. ajustado con LoRA (juez Groq, 1 a 5)")
ax.set_ylabel("Promedio")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

display(resultados.pivot(index="tema", columns="modelo", values="exactitud"))

## 7. Guardar el adaptador

El adaptador LoRA pesa unos pocos MB, porque solo contiene las matrices entrenadas y no el modelo completo. Se puede subir a Hugging Face para usarlo fuera de Colab.

In [ ]:
# Save the LoRA adapter (optionally push it to the Hugging Face Hub)
modelo_lora.save_pretrained("/content/dulcebot-tinyllama-lora")
tokenizer.save_pretrained("/content/dulcebot-tinyllama-lora")
print("Adaptador guardado en /content/dulcebot-tinyllama-lora")

# modelo_lora.push_to_hub("tu-usuario/dulcebot-tinyllama-lora")
# tokenizer.push_to_hub("tu-usuario/dulcebot-tinyllama-lora")

In [ ]:
# Try the full pipeline (RAG + fine-tuned Llama) with your own question
pregunta = "¿Puedo pasar el domingo a las 4 de la tarde por mi pastel?"
print("Contexto recuperado:", buscar_contexto(pregunta)[:150], "…")
print("DulceBot (LoRA):", generar_respuesta(modelo_lora, pregunta))

## 8. Conclusiones y decisión técnica

**Cómo leer los resultados**
- Si LoRA sube en **tono** y **formato**, el ajuste enseñó la voz de DulceBot: respuestas cortas, cálidas, sin emojis y con cierre de ayuda. Eso es lo que mejor aprende un fine-tuning con pocos ejemplos.
- La **exactitud** depende sobre todo del **RAG**. Si el modelo base ya acierta los datos cuando el contexto es correcto, eso confirma que los hechos deben venir de la recuperación y no de los pesos.
- **Fluidez no es exactitud:** lee las respuestas, no solo las notas. En la primera corrida, el modelo ajustado inventó con total seguridad un envío de $15, un WhatsApp, descuentos y el clima, todo con la voz de DulceBot. Un modelo pequeño ajustado suena convincente aunque se equivoque.
- Revisa los temas donde LoRA baja: suelen ser preguntas con un razonamiento extra (por ejemplo, "en dos meses" contra un máximo de 30 días). Ahí un modelo de 1.1B tiene límites.

**Por qué en producción DulceBot usa RAG + function calling con `gpt-oss-20b` y no este modelo**
1. **Datos vivos:** precios, stock y pedidos cambian a diario. Las herramientas (`buscar_en_menu`, `consultar_stock`, `consultar_pedido`) consultan la base de datos en tiempo real, y un modelo ajustado no puede hacerlo.
2. **Acciones:** DulceBot agrega productos al carrito, inicia encargos y cambia estatus. Eso requiere *function calling* fiable, que un modelo de 1.1B no ofrece.
3. **Infraestructura:** servir TinyLlama necesita una GPU dedicada, y el plan gratuito de Groq ya no incluye modelos Llama conversacionales. Por eso producción usa `gpt-oss-20b` en Groq, el mismo modelo que usa el curso en la Clase 4.

**Cuándo sí convendría el modelo ajustado:** en un despliegue propio (por ejemplo, un endpoint con GPU o un modelo Llama 3 de 8B con este mismo adaptador), para fijar tono y políticas sin un system prompt tan largo, lo que reduce los tokens por mensaje.